# Project 04: First Machine Learning Classifier
**InternCircle | Data Science & AI**

## 1. Objective
Predict whether a Titanic passenger survived using passenger class, sex, age and fare. The target is `survived`; the input features are `pclass`, `sex`, `age` and `fare`.

## 2. Dataset
This notebook uses the Titanic dataset through Seaborn's `sns.load_dataset("titanic")`, as specified in the learner guide. It keeps only the five required columns and removes rows with missing values. Internet access is required the first time the dataset is downloaded.

## 3. Workflow
1. Load and inspect the data.
2. Prepare features and target.
3. Split into training and test sets (80/20, random state 42, stratified).
4. Calculate a majority-class baseline.
5. Train a Decision Tree Classifier with maximum depth 4.
6. Evaluate test accuracy, confusion matrix and mean squared error on binary predictions.
7. Compare training and test accuracy for tree depths 1 to 12.
8. Review feature importance, findings and limitations.

## 4. Import libraries and load the Titanic dataset

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Load the dataset specified in the project guide
titanic = sns.load_dataset("titanic")

df = titanic[["survived", "pclass", "sex", "age", "fare"]].dropna().copy()
df["sex"] = (df["sex"] == "female").astype(int)

print("Dataset shape after selecting columns and dropping missing values:", df.shape)
display(df.head())
print("\\nMissing values:\\n", df.isnull().sum())
print("\\nTarget distribution:\\n", df["survived"].value_counts())

## 5. Define features and target

In [ ]:
X = df.drop(columns="survived")
y = df["survived"]

print("Prediction target: survived (0 = did not survive, 1 = survived)")
print("Features:", X.columns.tolist())
display(X.describe().round(2))

## 6. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print(f"Survival rate in training set: {y_train.mean():.3f}")
print(f"Survival rate in test set: {y_test.mean():.3f}")

## 7. Majority-class baseline
The baseline always predicts the most common class in the training data. Its accuracy is measured on the held-out test set. The model should perform better than this simple benchmark to add predictive value.

In [ ]:
majority_class = y_train.mode()[0]
baseline_predictions = np.full(shape=len(y_test), fill_value=majority_class)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Majority class selected from training data:", majority_class)
print(f"Baseline test accuracy: {baseline_accuracy:.3f}")

## 8. Train the Decision Tree Classifier

In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
train_accuracy = accuracy_score(y_train, model.predict(X_train))
test_accuracy = accuracy_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Baseline test accuracy: {baseline_accuracy:.3f}")
print(f"Mean squared error on 0/1 predictions: {mse:.3f}")
print("\\nClassification report:\\n")
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))

## 9. Confusion matrix
For the matrix layout below, rows are actual classes and columns are predicted classes:
- **True Negative (top-left):** actually did not survive, predicted did not survive.
- **False Positive (top-right):** actually did not survive, predicted survived.
- **False Negative (bottom-left):** actually survived, predicted did not survive.
- **True Positive (bottom-right):** actually survived, predicted survived.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Confusion matrix (rows = actual, columns = predicted):\\n", cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did not survive", "Survived"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Decision Tree — Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 10. Test tree depth and inspect overfitting
Train models with maximum depth from 1 to 12. A large gap between training and test accuracy can indicate overfitting.

In [ ]:
depth_results = []

for depth in range(1, 13):
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    depth_model.fit(X_train, y_train)

    depth_results.append({
        "max_depth": depth,
        "training_accuracy": accuracy_score(y_train, depth_model.predict(X_train)),
        "test_accuracy": accuracy_score(y_test, depth_model.predict(X_test))
    })

depth_results = pd.DataFrame(depth_results)
display(depth_results.round(3))

plt.figure(figsize=(9, 5))
plt.plot(depth_results["max_depth"], depth_results["training_accuracy"], marker="o", label="Training accuracy")
plt.plot(depth_results["max_depth"], depth_results["test_accuracy"], marker="o", label="Test accuracy")
plt.xlabel("Maximum tree depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Depth: Training vs Test Accuracy")
plt.xticks(range(1, 13))
plt.legend()
plt.tight_layout()
plt.show()

## 11. Feature importance

In [ ]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(feature_importance.rename("importance").to_frame().round(3))

feature_importance.sort_values().plot(kind="barh", figsize=(8, 4), title="Decision Tree Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 12. Results summary (generated when you run the notebook)
This cell collects the main values for the project write-up.

In [ ]:
print("PROJECT 04 — RESULTS SUMMARY")
print("-" * 40)
print(f"Rows used after cleaning: {len(df)}")
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")
print(f"Baseline test accuracy: {baseline_accuracy:.3f}")
print(f"Decision Tree training accuracy: {train_accuracy:.3f}")
print(f"Decision Tree test accuracy: {test_accuracy:.3f}")
print(f"Test mean squared error: {mse:.3f}")
print(f"Model beats baseline: {test_accuracy > baseline_accuracy}")
print("Most important feature:", feature_importance.index[0])
print("Most important feature score:", round(feature_importance.iloc[0], 3))
best_depth_row = depth_results.loc[depth_results["test_accuracy"].idxmax()]
print(f"Depth with highest test accuracy in this sweep: {int(best_depth_row['max_depth'])}")
print(f"Highest test accuracy in depth sweep: {best_depth_row['test_accuracy']:.3f}")

## 13. What was learned, limitations and conclusion

### What was learned
- A train/test split helps measure performance on data not used for fitting.
- A majority-class baseline provides a simple reference point.
- Accuracy and a confusion matrix reveal different aspects of classifier performance.
- Comparing train and test accuracy across tree depths helps identify overfitting.
- Feature importance provides a model-specific indication of which inputs the tree used most.

### Limitations
- Rows with missing values are removed, so the model is trained on a reduced subset of the Titanic data.
- The dataset is relatively small and historical; the results should not be generalized to other populations or modern travel.
- Accuracy alone can hide uneven performance between survival classes; the confusion matrix and classification report should also be reviewed.
- Decision-tree feature importance is not proof of causation.
- A deeper tree may fit training data closely but perform worse on unseen data.

### Conclusion
The notebook builds and evaluates a supervised classification pipeline using a Decision Tree. Use the actual values printed in the Results Summary cell when completing the short write-up. Do not claim the model is useful unless its test accuracy exceeds the baseline and the confusion matrix supports that conclusion.